In [ ]:
!pip install "transformers==4.39.3" "sentence-transformers>=2.7.0"


In [ ]:
import os
import json
import numpy as np
import pandas as pd
import traceback

from tqdm import tqdm
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity


DATASET_PATH = "campbell_qa.jsonl"
OUTPUT_FILE = "campbell_embedding_results.csv"

MODELS = [
    "intfloat/multilingual-e5-large",
    "sentence-transformers/paraphrase-multilingual-mpnet-base-v2",
]

TOP_K = [1, 3, 5, 10]

def load_dataset(path):
    data = []
    with open(path, "r", encoding="utf-8") as f:
        for line_number, line in enumerate(f, 1):
            line = line.strip()
            if not line:
                continue
            try:
                item = json.loads(line)
                data.append(item)
            except json.JSONDecodeError as e:
                print(f"JSON error in line {line_number}: {e}")
    print(f"Loaded {len(data)} questions.")
    return data

def prepare_dataset(data):
    queries = []
    candidate_documents = []
    relevant_indices = []
    for item in data:
        question = item.get("question", "").strip()
        options = item.get("options", {})
        correct_answer = item.get("answer", "").strip().lower()
        answer_text = item.get("answer_text", "").strip()
        explanation = item.get("explanation", "").strip()

        if not question or not isinstance(options, dict):
            continue

        if correct_answer not in options:
            print(f"Warning: invalid answer for {item.get('id')}")
            continue

        queries.append(question)
        docs = []
        correct_index = None

        for idx, (label, option_text) in enumerate(options.items()):
            option_text = str(option_text).strip()
            # Correct answer
            if label.lower() == correct_answer:
                document = f"Answer: {option_text}\nExplanation: {explanation}"
                correct_index = idx
            # Incorrect answer
            else:
                document = option_text
            docs.append(document)

        candidate_documents.append(docs)
        relevant_indices.append(correct_index)

    print(f"Prepared {len(queries)} benchmark queries.")
    return (queries, candidate_documents, relevant_indices)

def recall_at_k(ranked_indices, relevant_index, k):
    return float(relevant_index in ranked_indices[:k])

def reciprocal_rank_at_k(ranked_indices, relevant_index, k):
    for rank, idx in enumerate(ranked_indices[:k], start=1):
        if idx == relevant_index:
            return 1.0 / rank
    return 0.0

def ndcg_at_k(ranked_indices, relevant_index, k):
    for rank, idx in enumerate(ranked_indices[:k], start=1):
        if idx == relevant_index:
            return 1.0 / np.log2(rank + 1)
    return 0.0

def evaluate_model(model_name, queries, candidate_documents, relevant_indices):
    print("\n" + "=" * 70)
    print(f"Model: {model_name}")
    print("=" * 70)
    model = SentenceTransformer(model_name, trust_remote_code=True)

    model.max_seq_length = 512
    model_lower = model_name.lower()
    query_prefix = ""
    doc_prefix = ""

    if "e5" in model_lower:
        query_prefix = "query: "
        doc_prefix = "passage: "
    elif "bge" in model_lower or "arctic" in model_lower:
        query_prefix = "Represent this sentence for searching relevant passages: "
    elif "nomic" in model_lower:
        query_prefix = "search_query: "
        doc_prefix = "search_document: "

    prefixed_queries = [query_prefix + q for q in queries]
    flat_docs = []
    doc_group_sizes = []
    for docs in candidate_documents:
        doc_group_sizes.append(len(docs))
        for d in docs:
            flat_docs.append(doc_prefix + d)

    print("Encoding queries...")
    query_embeddings = model.encode(prefixed_queries, batch_size=32, normalize_embeddings=True, convert_to_numpy=True, show_progress_bar=True)

    print("Encoding candidate documents...")
    flat_doc_embeddings = model.encode(flat_docs, batch_size=32, normalize_embeddings=True, convert_to_numpy=True, show_progress_bar=True)

    metrics = {}
    for k in TOP_K:
        metrics[f"Recall@{k}"] = []
        metrics[f"MRR@{k}"] = []
        metrics[f"nDCG@{k}"] = []

    doc_idx = 0
    for i, (q_emb, rel_idx) in enumerate(tqdm(zip(query_embeddings, relevant_indices), total=len(queries), desc="Evaluating / Scoring")):
        size = doc_group_sizes[i]
        d_embs = flat_doc_embeddings[doc_idx : doc_idx + size]
        doc_idx += size

        # Calculate cosine similarity
        scores = cosine_similarity(q_emb.reshape(1, -1), d_embs)[0]
        ranked_indices = np.argsort(scores)[::-1]

        for k in TOP_K:
            metrics[f"Recall@{k}"].append(recall_at_k(ranked_indices, rel_idx, k))
            metrics[f"MRR@{k}"].append(reciprocal_rank_at_k(ranked_indices, rel_idx, k))
            metrics[f"nDCG@{k}"].append(ndcg_at_k(ranked_indices, rel_idx, k))

    # Compile results
    result = {"model": model_name, "num_questions": len(queries)}
    for metric_name, values in metrics.items():
        result[metric_name] = np.mean(values)

    return result


def main():
    data = load_dataset(DATASET_PATH)
    (queries, candidate_documents, relevant_indices) = prepare_dataset(data)
    if len(queries) == 0:
        raise RuntimeError("No benchmark queries were prepared. Check the dataset structure.")

    evaluated_models = set()
    if os.path.exists(OUTPUT_FILE):
        try:
            existing_df = pd.read_csv(OUTPUT_FILE)
            if 'model' in existing_df.columns:
                evaluated_models = set(existing_df['model'].tolist())
                print(f"\n[INFO] Found existing progress. Skipping already evaluated models: {evaluated_models}")
        except Exception as e:
            print(f"[WARNING] Could not read existing CSV: {e}")

    for model_name in MODELS:
        if model_name in evaluated_models:
            print(f"\n--- Skipping '{model_name}' (Already evaluated) ---")
            continue

        try:
            result = evaluate_model(model_name, queries, candidate_documents, relevant_indices)
            df_result = pd.DataFrame([result])
            if not os.path.exists(OUTPUT_FILE):
                df_result.to_csv(OUTPUT_FILE, index=False, encoding="utf-8-sig")
            else:
                df_result.to_csv(OUTPUT_FILE, mode='a', header=False, index=False, encoding="utf-8-sig")

            print(f"Results for '{model_name}' saved to {OUTPUT_FILE}.")

        except Exception as e:
            print(f"Error occurred while evaluating {model_name}: {e}")
            traceback.print_exc()
            continue

  
    if os.path.exists(OUTPUT_FILE):
        final_df = pd.read_csv(OUTPUT_FILE)

        
        if "MRR@5" in final_df.columns:
            final_df = final_df.sort_values(by="MRR@5", ascending=False)

        print("\n")
        print("=" * 100)
        print("FINAL RESULTS")
        print("=" * 100)

        
        def format_float(val):
            if isinstance(val, float):
                return f"{val:.4f}"
            return val

        print(final_df.to_string(index=False, formatters={col: format_float for col in final_df.select_dtypes(include=['float']).columns}))
        print(f"\nAll results are successfully stored in: {OUTPUT_FILE}")
    else:
        print("\nNo results were generated. Check for errors.")

if __name__ == "__main__":
    main()


Loaded 232 questions.
Prepared 231 benchmark queries.

Model: intfloat/multilingual-e5-large


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Encoding queries...


Batches:   0%|          | 0/8 [00:00<?, ?it/s]

Encoding candidate documents...


...

Batches:   0%|          | 0/8 [00:00<?, ?it/s]

Encoding candidate documents...


Batches:   0%|          | 0/36 [00:00<?, ?it/s]

Evaluating / Scoring: 100%|██████████| 231/231 [00:00<00:00, 2035.73it/s]

✅ Results for 'sentence-transformers/paraphrase-multilingual-mpnet-base-v2' saved to campbell_embedding_results.csv.


FINAL RESULTS
                                                      model  num_questions Recall@1  MRR@1 nDCG@1 Recall@3  MRR@3 nDCG@3 Recall@5  MRR@5 nDCG@5 Recall@10 MRR@10 nDCG@10
                             intfloat/multilingual-e5-large            231   0.9437 0.9437 0.9437   0.9870 0.9625 0.9688   1.0000 0.9657 0.9744    1.0000 0.9657  0.9744
sentence-transformers/paraphrase-multilingual-mpnet-base-v2            231   0.8571 0.8571 0.8571   0.9784 0.9113 0.9285   1.0000 0.9165 0.9377    1.0000 0.9165  0.9377

All results are successfully stored in: campbell_embedding_results.csv
